In [ ]:
"""
Task 4: Location-based Analysis

- Visualize restaurant distribution on a lat/lon map.
- Group by city / locality: concentration, average rating, price, cuisines.
- Surface geographic patterns and insights.

Outputs plots to ./task4_outputs/ and prints summary tables.
"""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt

DATA_PATH = Path(r"D:\Code\Repo\Machine-Learning-Internship\Dataset .csv")
OUT_DIR = Path(r"D:\Code\Repo\Machine-Learning-Internship\task4_outputs")
OUT_DIR.mkdir(exist_ok=True)


In [ ]:
def load(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["Cuisines"] = df["Cuisines"].fillna("Unknown")
    # A handful of rows have Longitude=0, Latitude=0 — treat as missing.
    bad_coords = (df["Longitude"] == 0) & (df["Latitude"] == 0)
    df.loc[bad_coords, ["Longitude", "Latitude"]] = np.nan
    return df


# ---------- Plots ----------


In [ ]:
def plot_world_map(df: pd.DataFrame):
    valid = df.dropna(subset=["Longitude", "Latitude"])
    fig, ax = plt.subplots(figsize=(12, 6))
    sc = ax.scatter(valid["Longitude"], valid["Latitude"],
                    c=valid["Aggregate rating"], cmap="viridis",
                    s=6, alpha=0.5)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title("Restaurants worldwide (colored by aggregate rating)")
    plt.colorbar(sc, ax=ax, label="Aggregate rating")
    fig.tight_layout(); fig.savefig(OUT_DIR / "map_world.png", dpi=120)
    plt.close(fig)


In [ ]:
def plot_india_zoom(df: pd.DataFrame):
    india = df[(df["Longitude"].between(68, 98)) &
               (df["Latitude"].between(6, 36))]
    if india.empty:
        return
    fig, ax = plt.subplots(figsize=(10, 9))
    sc = ax.scatter(india["Longitude"], india["Latitude"],
                    c=india["Aggregate rating"], cmap="viridis",
                    s=6, alpha=0.5)
    ax.set_title(f"India zoom — {len(india)} restaurants "
                 f"(colored by aggregate rating)")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    plt.colorbar(sc, ax=ax, label="Aggregate rating")
    fig.tight_layout(); fig.savefig(OUT_DIR / "map_india.png", dpi=120)
    plt.close(fig)


In [ ]:
def plot_top_cities(city_stats: pd.DataFrame):
    top = city_stats.head(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top.index[::-1], top["count"][::-1], color="steelblue")
    ax.set_title("Top 15 cities by restaurant count")
    ax.set_xlabel("Number of restaurants")
    fig.tight_layout(); fig.savefig(OUT_DIR / "top_cities.png", dpi=120)
    plt.close(fig)


In [ ]:
def plot_rating_vs_count(city_stats: pd.DataFrame):
    sig = city_stats[city_stats["count"] >= 20]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(sig["count"], sig["avg_rating"], s=30, alpha=0.6)
    ax.set_xscale("log")
    ax.set_xlabel("Number of restaurants (log)")
    ax.set_ylabel("Average aggregate rating")
    ax.set_title("City scale vs. average rating (cities with >=20 restaurants)")
    # Annotate a few notable points
    for city in ["New Delhi", "Bangalore", "Gurgaon", "Noida",
                 "Ghaziabad", "Faridabad"]:
        if city in sig.index:
            ax.annotate(city, (sig.loc[city, "count"],
                               sig.loc[city, "avg_rating"]),
                        fontsize=8, xytext=(4, 2), textcoords="offset points")
    fig.tight_layout(); fig.savefig(OUT_DIR / "rating_vs_count.png", dpi=120)
    plt.close(fig)


# ---------- Stats ----------


In [ ]:
def city_stats(df: pd.DataFrame) -> pd.DataFrame:
    # Drop unrated (0) from rating aggregations — they bias averages downward.
    rated = df[df["Aggregate rating"] > 0]
    g_all = df.groupby("City")
    g_rated = rated.groupby("City")

    def top_cuisine(series: pd.Series) -> str:
        ctr = Counter()
        for s in series.dropna():
            for c in s.split(","):
                ctr[c.strip()] += 1
        return ctr.most_common(1)[0][0] if ctr else "—"

    stats = pd.DataFrame({
        "count":      g_all.size(),
        "avg_rating": g_rated["Aggregate rating"].mean(),
        "rated_pct":  g_rated.size() / g_all.size() * 100,
        "avg_cost":   g_all["Average Cost for two"].mean(),
        "avg_price_range": g_all["Price range"].mean(),
        "top_cuisine":     g_all["Cuisines"].apply(top_cuisine),
    }).sort_values("count", ascending=False)
    return stats


In [ ]:
def locality_stats(df: pd.DataFrame) -> pd.DataFrame:
    rated = df[df["Aggregate rating"] > 0]
    g_all = df.groupby(["City", "Locality"])
    g_rated = rated.groupby(["City", "Locality"])
    stats = pd.DataFrame({
        "count":      g_all.size(),
        "avg_rating": g_rated["Aggregate rating"].mean(),
        "avg_cost":   g_all["Average Cost for two"].mean(),
    })
    return stats


# ---------- Main ----------


In [ ]:
def main():
    df = load(DATA_PATH)
    print(f"Loaded {len(df)} restaurants "
          f"({df[['Longitude','Latitude']].dropna().shape[0]} with coords)")

    # Country spread (via country code)
    print("\n=== Restaurants per country code (top 10) ===")
    print(df["Country Code"].value_counts().head(10).to_string())

    # City stats
    cs = city_stats(df)
    print("\n=== Top 10 cities by restaurant count ===")
    print(cs.head(10).round(2).to_string())

    print("\n=== Best-rated cities (>=30 restaurants, >=50% rated) ===")
    eligible = cs[(cs["count"] >= 30) & (cs["rated_pct"] >= 50)]
    print(eligible.sort_values("avg_rating", ascending=False)
                  .head(10).round(2).to_string())

    print("\n=== Most expensive cities (>=20 restaurants) ===")
    print(cs[cs["count"] >= 20].sort_values("avg_cost", ascending=False)
                               .head(10).round(0).to_string())

    # Locality stats
    ls = locality_stats(df)
    print("\n=== Top 10 localities by restaurant density ===")
    print(ls.sort_values("count", ascending=False).head(10).round(2).to_string())

    print("\n=== Highest-rated localities (>=20 restaurants) ===")
    print(ls[ls["count"] >= 20].sort_values("avg_rating", ascending=False)
                               .head(10).round(2).to_string())

    # Price-range distribution globally
    print("\n=== Price-range distribution ===")
    print(df["Price range"].value_counts().sort_index().to_string())

    # Save plots
    plot_world_map(df)
    plot_india_zoom(df)
    plot_top_cities(cs)
    plot_rating_vs_count(cs)
    print(f"\nPlots written to: {OUT_DIR}")

    # Save full tables too
    cs.to_csv(OUT_DIR / "city_stats.csv")
    ls.to_csv(OUT_DIR / "locality_stats.csv")
    print(f"Stats CSVs written to: {OUT_DIR}")


In [ ]:
if __name__ == "__main__":
    main()
